# PlotPot SPR kinetics notebook (Colab version)

Fits and plots 1:1 Langmuir binding kinetics from SPR (or BLI) sensorgram data.

**Input format:** Tab-separated export from Biacore or similar.  
Auto-detects two layouts:
- **Wide:** `Time | Chan1 | Chan2 | ...` — one time column, one response column per sensorgram  
- **Paired:** `Time | Response | Time | Response | ...` — alternating pairs per sensorgram

**Kinetic methods:**
- **SCK** (single-cycle kinetics) — cycles injected sequentially without regeneration; model chains cycles together
- **MCK** (multi-cycle kinetics) — each cycle starts from a regenerated baseline; cycles fitted independently

**Workflow** — run cells top to bottom:
1. **Dependencies · Imports** — run once.
2. **Upload** — select your SPR export `.txt`.
3. **Experiment setup** — concentrations, time windows, method.
4. **Overview** — raw sensorgrams with association/dissociation markers.
5. **Fit** — global 1:1 kinetics; prints ka, kd, KD, Rmax.
6. **Publication plot** — data (dots) + fit (line) per concentration.
7. **Save & download** — PDF + PNG.

In [ ]:
#@title Step 1 · Install & verify dependencies { display-mode: "form" }
import importlib, subprocess, sys

_required = ['numpy', 'pandas', 'matplotlib', 'scipy']
_missing  = [p for p in _required if importlib.util.find_spec(p) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + _missing)
    print(f'Installed: {_missing}')
else:
    print('All dependencies present:', _required)

In [ ]:
#@title Step 2 · Imports & plot style { display-mode: "form" }
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from pathlib import Path
from scipy.optimize import least_squares

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 10,
    'axes.linewidth': 0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'xtick.direction': 'out',
    'ytick.direction': 'out',
    'pdf.fonttype': 42,
    'svg.fonttype': 'none',
})

In [ ]:
#@title Step 3 · Upload data { display-mode: "form" }
from google.colab import files as _colab_files

print('Select your SPR export (.txt):')
_uploaded = _colab_files.upload()

_fname    = next(iter(_uploaded))
DATA_FILE = Path(_fname)
_raw_text = _uploaded[_fname].decode('utf-8')


def _is_numeric(s):
    try:
        float(s.replace(',', '.'))
        return True
    except ValueError:
        return False


def load_spr_export(text):
    """
    Load an SPR export file and return a dict of {name: (time, response)}.

    Handles two layouts:
      Wide  : Time | Chan1 | Chan2 | ...  (first column is time, rest are responses)
      Paired: Time | Response | Time | Response | ...  (alternating pairs)

    Skips any header rows where the first cell is non-numeric (e.g. units rows).
    """
    lines = [l for l in text.splitlines() if l.strip()]

    # Separate header line(s) from data
    header_line = None
    data_lines  = []
    for line in lines:
        first = line.split('\t')[0].strip()
        if _is_numeric(first):
            data_lines.append(line)
        elif header_line is None:
            header_line = line
        # skip additional non-numeric rows (e.g. units)

    if not data_lines:
        raise ValueError('No numeric data rows found. Check file format.')

    headers = header_line.split('\t') if header_line else []
    rows = [l.split('\t') for l in data_lines]
    n_cols = max(len(r) for r in rows)
    # Pad short rows
    rows = [r + [''] * (n_cols - len(r)) for r in rows]

    def _f(s):
        try:
            return float(s.strip().replace(',', '.'))
        except:
            return np.nan

    arr = np.array([[_f(v) for v in r] for r in rows])  # shape (n_rows, n_cols)

    # Detect layout: paired if every even column (0, 2, 4...) is nearly identical
    # (same time axis repeated). Check first two even-column pairs.
    paired = False
    if n_cols >= 4 and n_cols % 2 == 0:
        t0 = arr[:, 0]
        t2 = arr[:, 2]
        if np.nanmean(np.abs(t0 - t2)) < 1e-6 * (np.nanmax(t0) - np.nanmin(t0) + 1e-9):
            paired = True

    sensorgrams = {}
    if paired:
        n_pairs = n_cols // 2
        for i in range(n_pairs):
            t = arr[:, 2 * i]
            r = arr[:, 2 * i + 1]
            mask = ~np.isnan(t) & ~np.isnan(r)
            name = headers[2 * i + 1] if 2 * i + 1 < len(headers) else f'Chan{i+1}'
            name = name.strip() or f'Chan{i+1}'
            sensorgrams[name] = (t[mask], r[mask])
    else:
        t = arr[:, 0]
        for i in range(1, n_cols):
            r = arr[:, i]
            mask = ~np.isnan(t) & ~np.isnan(r)
            name = headers[i] if i < len(headers) else f'Chan{i}'
            name = name.strip() or f'Chan{i}'
            sensorgrams[name] = (t[mask], r[mask])

    return sensorgrams, 'paired' if paired else 'wide'


sensorgrams, _layout = load_spr_export(_raw_text)
_chan_names = list(sensorgrams.keys())

print(f'Loaded {_fname!r}  (layout: {_layout})')
print(f'{len(sensorgrams)} sensorgram(s) found:\n')
for name, (t, r) in sensorgrams.items():
    print(f'  • {name!r}  t={t.min():.1f}–{t.max():.1f} s  '
          f'R={r.min():.2f}–{r.max():.2f} RU  (n={len(t):,})')
print()
print('Copy channel name(s) into CHANNELS in Step 4.')

In [ ]:
#@title Step 4 · Experiment setup { display-mode: "form" }
#@markdown **Channel selection** — comma-separated channel names from Step 3 output.
#@markdown Leave empty to use the first channel.
CHANNELS = "" #@param {type:"string"}

#@markdown ---
#@markdown **Concentrations** — comma-separated, in the order of injection (lowest to highest).
CONC_VALUES = "0.617, 1.85, 5.56, 16.7, 50" #@param {type:"string"}
CONC_UNIT   = "nM" #@param ["pM", "nM", "uM", "mM"]

#@markdown **Association start times (s)** — one per concentration.
T_ASS  = "35, 205, 375, 545, 715"  #@param {type:"string"}
#@markdown **Dissociation start times (s)** — one per concentration.
T_DISS = "145, 315, 485, 655, 825" #@param {type:"string"}

#@markdown ---
#@markdown **Kinetic method**
METHOD = "SCK" #@param ["SCK", "MCK"]

#@markdown ---
#@markdown **Crop time window (s)** — leave 0/0 to use the full trace.
T_START = 0.0  #@param {type:"number"}
T_END   = 0.0  #@param {type:"number"}

# ── Resolve ────────────────────────────────────────────────────────────────────
_unit_factors = {'pM': 1e-12, 'nM': 1e-9, 'uM': 1e-6, 'mM': 1e-3}

def _csv_floats(s):
    return [float(x.strip()) for x in s.split(',') if x.strip()]

_conc_raw  = _csv_floats(CONC_VALUES)
_conc_M    = [c * _unit_factors[CONC_UNIT] for c in _conc_raw]
_t_ass     = _csv_floats(T_ASS)
_t_diss    = _csv_floats(T_DISS)
n_cycles   = len(_conc_M)

if CHANNELS.strip():
    _sel_chans = [c.strip() for c in CHANNELS.split(',') if c.strip() in sensorgrams]
    _missing   = [c.strip() for c in CHANNELS.split(',') if c.strip() not in sensorgrams]
    if _missing:
        print(f'WARNING: channels not found: {_missing}')
else:
    _sel_chans = [_chan_names[0]]

assert len(_t_ass)  == n_cycles, f'Need {n_cycles} t_ass values, got {len(_t_ass)}'
assert len(_t_diss) == n_cycles, f'Need {n_cycles} t_diss values, got {len(_t_diss)}'

print(f'Method   : {METHOD}')
print(f'Channels : {_sel_chans}')
print(f'Cycles   : {n_cycles}')
for i, (c, ta, td) in enumerate(zip(_conc_raw, _t_ass, _t_diss)):
    print(f'  Cycle {i+1}: {c} {CONC_UNIT}  ass={ta}s  diss={td}s')

In [ ]:
#@title Step 5 · Overview { display-mode: "form" }
n_ch  = len(_sel_chans)
fig_ov, axes_ov = plt.subplots(n_ch, 1, figsize=(9, 3.2 * n_ch), squeeze=False,
                                layout='constrained')

_COLORS = ['#1a4f8a', '#c0392b', '#27ae60', '#8e44ad', '#d35400',
           '#16a085', '#d4ac0d', '#7f8c8d']

for row, chan in enumerate(_sel_chans):
    ax  = axes_ov[row, 0]
    t, r = sensorgrams[chan]

    if T_START < T_END:
        mask = (t >= T_START) & (t <= T_END)
        t, r = t[mask], r[mask]

    ax.plot(t, r, color='#444', lw=0.8, zorder=2)

    # Mark association / dissociation windows
    y0, y1 = ax.get_ylim()
    for i, (ta, td) in enumerate(zip(_t_ass, _t_diss)):
        col = _COLORS[i % len(_COLORS)]
        ax.axvspan(ta, td, alpha=0.12, color=col, zorder=1)
        ax.axvline(ta,  color=col, lw=0.8, ls='--', alpha=0.7, zorder=3)
        ax.axvline(td, color=col, lw=0.8, ls=':',  alpha=0.7, zorder=3)
        ax.text((ta + td) / 2, ax.get_ylim()[1],
                f'{_conc_raw[i]} {CONC_UNIT}',
                ha='center', va='bottom', fontsize=7, color=col)

    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Response (RU)')
    ax.set_title(chan, fontsize=11, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig_ov.suptitle(f'{DATA_FILE.stem} — overview ({METHOD})', fontsize=12)
plt.show()

In [ ]:
#@title Step 6 · Global 1:1 kinetics fit { display-mode: "form" }
#@markdown *Re-run after changing parameters in Step 4.*

def _model_sck(t, ka, kd, Rmax, t_ass, t_diss, concs):
    """
    1:1 Langmuir SCK model. Cycles chain: dissociation of cycle i feeds R0 of cycle i+1.
    """
    R   = np.zeros_like(t, dtype=float)
    R0  = 0.0
    n   = len(concs)
    for i in range(n):
        C     = concs[i]
        kobs  = ka * C + kd
        Req   = Rmax * ka * C / kobs if kobs > 1e-30 else 0.0
        ta, td = t_ass[i], t_diss[i]
        t_next = t_ass[i + 1] if i + 1 < n else t[-1] + 1

        m_a = (t >= ta) & (t < td)
        if m_a.any():
            R[m_a] = Req + (R0 - Req) * np.exp(-kobs * (t[m_a] - ta))

        R_at_diss = Req + (R0 - Req) * np.exp(-kobs * (td - ta))

        m_d = (t >= td) & (t < t_next)
        if m_d.any():
            R[m_d] = R_at_diss * np.exp(-kd * (t[m_d] - td))

        R0 = R_at_diss * np.exp(-kd * (t_next - td))

    return R


def _model_mck(t, ka, kd, Rmax, t_ass, t_diss, concs):
    """
    1:1 Langmuir MCK model. Each cycle starts from R0=0 (regenerated baseline).
    """
    R = np.zeros_like(t, dtype=float)
    n = len(concs)
    for i in range(n):
        C     = concs[i]
        kobs  = ka * C + kd
        Req   = Rmax * ka * C / kobs if kobs > 1e-30 else 0.0
        ta, td = t_ass[i], t_diss[i]
        t_next = t_ass[i + 1] if i + 1 < n else t[-1] + 1

        m_a = (t >= ta) & (t < td)
        if m_a.any():
            R[m_a] = Req * (1.0 - np.exp(-kobs * (t[m_a] - ta)))

        R_at_diss = Req * (1.0 - np.exp(-kobs * (td - ta)))

        m_d = (t >= td) & (t < t_next)
        if m_d.any():
            R[m_d] = R_at_diss * np.exp(-kd * (t[m_d] - td))

    return R


_model_fn = _model_sck if METHOD == 'SCK' else _model_mck

# Collect data for fitting (all selected channels concatenated)
_t_fit, _r_fit = [], []
for chan in _sel_chans:
    t, r = sensorgrams[chan]
    if T_START < T_END:
        mask = (t >= T_START) & (t <= T_END)
        t, r = t[mask], r[mask]
    # Keep only points within the experiment window
    t_exp_start = _t_ass[0]
    t_exp_end   = _t_diss[-1] + (_t_ass[-1] - _t_diss[-2]) if n_cycles > 1 else _t_diss[-1] + 120
    mask2 = (t >= t_exp_start) & (t <= t_exp_end)
    _t_fit.append(t[mask2])
    _r_fit.append(r[mask2])

_t_all = np.concatenate(_t_fit)
_r_all = np.concatenate(_r_fit)

# Initial guesses
_Rmax0 = float(np.nanmax(_r_all)) * 1.5
_p0_log = [np.log10(1e5), np.log10(1e-3), _Rmax0]  # log10(ka), log10(kd), Rmax

def _residuals(p):
    ka   = 10 ** p[0]
    kd   = 10 ** p[1]
    Rmax = p[2]
    if Rmax <= 0:
        return np.full_like(_r_all, 1e9)
    pred = _model_fn(_t_all, ka, kd, Rmax, _t_ass, _t_diss, _conc_M)
    return pred - _r_all

_bounds_lo = [np.log10(1e2),  np.log10(1e-6), 0.0]
_bounds_hi = [np.log10(1e9),  np.log10(10.0), _Rmax0 * 10]

result_fit = least_squares(_residuals, _p0_log,
                           bounds=(_bounds_lo, _bounds_hi),
                           method='trf', max_nfev=5000)

ka_fit   = 10 ** result_fit.x[0]
kd_fit   = 10 ** result_fit.x[1]
Rmax_fit = result_fit.x[2]
KD_fit   = kd_fit / ka_fit

# Parameter uncertainties from covariance (Jacobian approximation)
try:
    J     = result_fit.jac
    cov   = np.linalg.pinv(J.T @ J) * (np.sum(_residuals(result_fit.x)**2) / (len(_r_all) - 3))
    errs  = np.sqrt(np.diag(np.abs(cov)))
    ka_se, kd_se = 10**result_fit.x[0] * np.log(10) * errs[0], 10**result_fit.x[1] * np.log(10) * errs[1]
    KD_se = KD_fit * np.sqrt((ka_se/ka_fit)**2 + (kd_se/kd_fit)**2)
    Rmax_se = errs[2]
except Exception:
    ka_se = kd_se = KD_se = Rmax_se = float('nan')

def _fmt_si(v, se=None):
    """Format a value with SI prefix."""
    prefixes = [(1e-12,'p'),(1e-9,'n'),(1e-6,'u'),(1e-3,'m'),(1,'')]
    for scale, pfx in reversed(prefixes):
        if v >= scale:
            s = f'{v/scale:.2g} {pfx}'
            if se and not np.isnan(se):
                s += f' ± {se/scale:.1g} {pfx}'
            return s
    return f'{v:.2e}'

print(f'=== Global 1:1 kinetics fit ({METHOD}) ===\n')
print(f'  ka   = {_fmt_si(ka_fit, ka_se)}M⁻¹s⁻¹')
print(f'  kd   = {_fmt_si(kd_fit, kd_se)}s⁻¹')
print(f'  KD   = {_fmt_si(KD_fit, KD_se)}M')
print(f'  Rmax = {Rmax_fit:.1f} ± {Rmax_se:.1f} RU' if not np.isnan(Rmax_se) else f'  Rmax = {Rmax_fit:.1f} RU')
print(f'\n  χ²/n = {np.sum(_residuals(result_fit.x)**2)/len(_r_all):.4f} RU²')
print(f'  Converged: {result_fit.success}  ({result_fit.message})')

In [ ]:
#@title Step 7 · Publication plot { display-mode: "form" }
#@markdown *Re-run Step 6 first if you changed parameters.*

C_DATA = '#888888'
C_FIT  = '#1a4f8a'

ncols  = min(n_cycles, 3)
nrows  = (n_cycles + ncols - 1) // ncols

fig, axes_p = plt.subplots(nrows, ncols,
                            figsize=(3.8 * ncols, 3.4 * nrows),
                            layout='constrained')
axes_flat = np.array(axes_p).flatten()

for i, (C_M, c_val) in enumerate(zip(_conc_M, _conc_raw)):
    ax  = axes_flat[i]
    ta, td = _t_ass[i], _t_diss[i]
    t_next = _t_ass[i + 1] if i + 1 < n_cycles else td + (_t_ass[-1] - _t_ass[0]) / max(n_cycles - 1, 1)

    # Data (all selected channels averaged if multiple)
    t_ref = None
    r_stack = []
    for chan in _sel_chans:
        t_c, r_c = sensorgrams[chan]
        mask = (t_c >= ta) & (t_c <= t_next)
        if t_ref is None:
            t_ref = t_c[mask]
        r_interp = np.interp(t_ref, t_c[mask], r_c[mask]) if len(t_c[mask]) else np.full_like(t_ref, np.nan)
        r_stack.append(r_interp)

    r_mean = np.nanmean(r_stack, axis=0)
    t_plot = t_ref - ta  # zero at injection start

    ax.scatter(t_plot, r_mean, color=C_DATA, s=4, alpha=0.6, zorder=2, label='Data')

    # Fit curve
    t_fit = np.linspace(ta, t_next, 500)
    r_fit = _model_fn(t_fit, ka_fit, kd_fit, Rmax_fit, _t_ass, _t_diss, _conc_M)
    # Slice out cycle i window
    mask_fit = (t_fit >= ta) & (t_fit <= t_next)
    ax.plot(t_fit[mask_fit] - ta, r_fit[mask_fit], color=C_FIT, lw=1.5, zorder=3, label='Fit')

    # Dissociation marker
    ax.axvline(td - ta, color='#aaa', lw=0.8, ls='--', zorder=1)

    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Response (RU)')
    ax.set_title(f'{c_val} {CONC_UNIT}', fontsize=10, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    if i == 0:
        ax.legend(loc='upper right', frameon=False, fontsize=8)

# Hide unused axes
for j in range(n_cycles, len(axes_flat)):
    axes_flat[j].set_visible(False)

fig.suptitle(
    f'{DATA_FILE.stem}\n'
    f'ka={ka_fit:.2e} M⁻¹s⁻¹  kd={kd_fit:.2e} s⁻¹  KD={_fmt_si(KD_fit)}M  Rmax={Rmax_fit:.0f} RU',
    fontsize=10
)
plt.show()

In [ ]:
#@title Step 8 · Save & download { display-mode: "form" }
from google.colab import files as _colab_files

OUTPUT_STEM = DATA_FILE.stem

_outputs = []
for ext in ('pdf', 'png'):
    out = f'/content/{OUTPUT_STEM}_spr_kinetics.{ext}'
    fig.savefig(out, dpi=300, bbox_inches='tight')
    print(f'Saved → {out}')
    _outputs.append(out)

print('\nStarting downloads...')
for out in _outputs:
    _colab_files.download(out)